# EDA 2 : Deep Dives

Analyses ciblées on les pistes à fort potentiel identifiées in l'EDA 1.

**Points d'approfondissement :**
1. Spark spread — non-linéarité et régimes
2. Thermal need × Gas price — interaction clé
3. Difficulté saisonnière — forquoi juillet-août sont si durs
4. Nucléaire FR — lag, seuils, interactions
5. UK wind — seuils de prix négatifs, non-linéarité
6. Régimes de prix — crise vs normal
7. Congestion interconnecteurs — découplage des marchés

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

from src.data_loading import load_data, merge_train

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['figure.dpi'] = 110

x_train, y_train, x_test = load_data()
train = merge_train(x_train, y_train)
dt = train['datetime_CET']

# Forward-fill daily columns
DAILY_COLS = ['de_gas', 'es_gas', 'fr_gas', 'nl_gas', 'uk_gas', 'eu_emission', 'uk_emission']
for col in DAILY_COLS:
    train[col] = train[col].ffill()

# Pre-compute key derived features
GAS_EFF = 0.50
EM_FACTOR = 0.37
train['fr_residual'] = train['fr_load_f'] - train['fr_solar_f'] - train['fr_wind_f']
train['uk_residual'] = train['uk_load_f'] - train['uk_solar_f'] - train['uk_wind_f']
train['fr_thermal_need'] = train['fr_residual'] - train['fr_nuclear_avcap_f']
train['uk_thermal_need'] = train['uk_residual'] - train['uk_nuclear_avcap_f']
train['fr_spark'] = train['fr_gas'] / GAS_EFF + train['eu_emission'] * EM_FACTOR
train['uk_spark'] = train['uk_gas'] / GAS_EFF + train['uk_emission'] * EM_FACTOR
train['fr_wind_pen'] = train['fr_wind_f'] / train['fr_load_f'].clip(lower=1)
train['uk_wind_pen'] = train['uk_wind_f'] / train['uk_load_f'].clip(lower=1)
train['spread_fr_uk'] = train['fr_spot'] - train['uk_spot']

print('Data loaded. Starting deep dives.')

---
## 1. Spark Spread — Le prédicteur roi (r=0.89)

Le spark spread approxime le coût marginal d'une centrale gaz. Quand le gaz fixe le prix marginal, spark ≈ spot price. Mais est-ce toujours le cas ?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

mask = train['fr_spark'].notna()
fr_s = train[mask]

# 1a. Scatter coloré par heure — le spark tient-il à toutes les heures ?
sc = axes[0, 0].scatter(fr_s['fr_spark'], fr_s['fr_spot'], c=fr_s['datetime_CET'].dt.hour,
                        cmap='twilight_shifted', s=2, alpha=0.4)
axes[0, 0].plot([0, 500], [0, 500], 'r--', linewidth=1)
axes[0, 0].set_xlabel('FR Spark Spread')
axes[0, 0].set_ylabel('FR Spot')
axes[0, 0].set_title('FR: Spark vs Spot (couleur = heure)')
plt.colorbar(sc, ax=axes[0, 0], label='Heure')

# 1b. Résidu spot - spark par heure
fr_s = fr_s.copy()
fr_s['spark_resid'] = fr_s['fr_spot'] - fr_s['fr_spark']
resid_by_hour = fr_s.groupby(fr_s['datetime_CET'].dt.hour)['spark_resid'].agg(['mean', 'std'])
axes[0, 1].bar(resid_by_hour.index, resid_by_hour['mean'], color='steelblue', alpha=0.7)
axes[0, 1].errorbar(resid_by_hour.index, resid_by_hour['mean'], yerr=resid_by_hour['std'],
                    fmt='none', color='black', capsize=2)
axes[0, 1].axhline(0, color='red', linewidth=0.5)
axes[0, 1].set_xlabel('Heure')
axes[0, 1].set_ylabel('Spot - Spark (EUR/MWh)')
axes[0, 1].set_title('FR: Biais du spark spread par heure')

# 1c. Corrélation spark-spot glissante (régimes)
window = 720  # 30 jours
mask_both = train['fr_spark'].notna()
rolling_r = train.loc[mask_both, 'fr_spot'].rolling(window).corr(train.loc[mask_both, 'fr_spark'])
axes[0, 2].plot(train.loc[mask_both, 'datetime_CET'], rolling_r, linewidth=1, color='#2166ac')
axes[0, 2].axhline(0.89, color='red', linestyle='--', label='r global = 0.89')
axes[0, 2].set_title('FR: Corrélation Spark-Spot glissante 30j')
axes[0, 2].set_ylabel('Pearson r')
axes[0, 2].legend()

# UK
mask = train['uk_spark'].notna()
uk_s = train[mask].copy()

sc = axes[1, 0].scatter(uk_s['uk_spark'], uk_s['uk_spot'], c=uk_s['datetime_CET'].dt.hour,
                        cmap='twilight_shifted', s=2, alpha=0.4)
axes[1, 0].plot([0, 500], [0, 500], 'r--', linewidth=1)
axes[1, 0].set_xlabel('UK Spark Spread')
axes[1, 0].set_ylabel('UK Spot')
axes[1, 0].set_title('UK: Spark vs Spot (couleur = heure)')
plt.colorbar(sc, ax=axes[1, 0], label='Heure')

uk_s['spark_resid'] = uk_s['uk_spot'] - uk_s['uk_spark']
resid_by_hour = uk_s.groupby(uk_s['datetime_CET'].dt.hour)['spark_resid'].agg(['mean', 'std'])
axes[1, 1].bar(resid_by_hour.index, resid_by_hour['mean'], color='#b2182b', alpha=0.7)
axes[1, 1].errorbar(resid_by_hour.index, resid_by_hour['mean'], yerr=resid_by_hour['std'],
                    fmt='none', color='black', capsize=2)
axes[1, 1].axhline(0, color='red', linewidth=0.5)
axes[1, 1].set_xlabel('Heure')
axes[1, 1].set_ylabel('Spot - Spark')
axes[1, 1].set_title('UK: Biais du spark spread par heure')

rolling_r = train.loc[mask, 'uk_spot'].rolling(window).corr(train.loc[mask, 'uk_spark'])
axes[1, 2].plot(train.loc[mask, 'datetime_CET'], rolling_r, linewidth=1, color='#b2182b')
axes[1, 2].axhline(0.88, color='red', linestyle='--', label='r global = 0.88')
axes[1, 2].set_title('UK: Corrélation Spark-Spot glissante 30j')
axes[1, 2].set_ylabel('Pearson r')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

# Analyse : quand le spark rate
print('Quand le spark spread ne fonctionne pas :')
print(f'\nFR — Spot << Spark (prix bas malgré gaz cher) :')
low_resid = fr_s[fr_s['spark_resid'] < -100]
print(f'  {len(low_resid)} heures ({100*len(low_resid)/len(fr_s):.1f}%)')
print(f'  Heure moyenne: {low_resid["datetime_CET"].dt.hour.mean():.0f}h')
print(f'  Wind penetration: {low_resid["fr_wind_pen"].mean()*100:.1f}% (vs normal {fr_s["fr_wind_pen"].mean()*100:.1f}%)')
print(f'  Nuclear: {low_resid["fr_nuclear_avcap_f"].mean():.0f} MW (vs normal {fr_s["fr_nuclear_avcap_f"].mean():.0f})')

print(f'\nFR — Spot >> Spark (prix bien au-dessus du gaz) :')
high_resid = fr_s[fr_s['spark_resid'] > 200]
print(f'  {len(high_resid)} heures ({100*len(high_resid)/len(fr_s):.1f}%)')
print(f'  Nuclear: {high_resid["fr_nuclear_avcap_f"].mean():.0f} MW (vs normal {fr_s["fr_nuclear_avcap_f"].mean():.0f})')
print(f'  Thermal need: {high_resid["fr_thermal_need"].mean():.0f} MW (vs normal {fr_s["fr_thermal_need"].mean():.0f})')

---
## 2. L'interaction magique : Thermal Need × Gas Price

Hypothèse : quand thermal_need > 0 (le gaz doit tourner), le prix suit gas_price. Quand thermal_need < 0 (assez de nucléaire+renouvelable), le prix décroche du gaz.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

mask = train['fr_gas'].notna()
sub = train[mask].copy()

# 2a. 3D view : thermal need × gas price → spot (heatmap binned)
sub['tn_bin'] = pd.cut(sub['fr_thermal_need'], bins=15)
sub['gas_bin'] = pd.cut(sub['fr_gas'], bins=10)
pivot = sub.pivot_table(values='fr_spot', index='gas_bin', columns='tn_bin', aggfunc='mean')

ax = axes[0, 0]
sns.heatmap(pivot, ax=ax, cmap='RdYlBu_r', annot=True, fmt='.0f', annot_kws={'size': 6})
ax.set_title('FR Spot moyen = f(Thermal Need, Gas Price)')
ax.set_xlabel('Thermal Need (MW)')
ax.set_ylabel('Gas Price (EUR/MWh)')

# 2b. Corrélation gas-spot conditionnelle au thermal need
ax = axes[0, 1]
tn_quantiles = sub['fr_thermal_need'].quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0])
colors = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 5))
for i in range(5):
    mask_q = (sub['fr_thermal_need'] >= tn_quantiles.iloc[i]) & (sub['fr_thermal_need'] < tn_quantiles.iloc[i+1])
    r = sub.loc[mask_q, 'fr_gas'].corr(sub.loc[mask_q, 'fr_spot'])
    label = f'TN q{i*20}-{(i+1)*20} (r={r:.2f})'
    ax.scatter(sub.loc[mask_q, 'fr_gas'], sub.loc[mask_q, 'fr_spot'], 
              s=2, alpha=0.3, color=colors[i], label=label)
ax.set_xlabel('FR Gas Price')
ax.set_ylabel('FR Spot')
ax.set_title('Gas→Spot dépend du Thermal Need')
ax.legend(fontsize=7, loc='upper left')

# 2c. Feature candidate : thermal_need × gas_price
sub['tn_x_gas'] = sub['fr_thermal_need'].clip(lower=0) * sub['fr_gas']
r_tn_gas = sub['tn_x_gas'].corr(sub['fr_spot'])
r_gas = sub['fr_gas'].corr(sub['fr_spot'])
r_tn = sub['fr_thermal_need'].corr(sub['fr_spot'])

ax = axes[0, 2]
ax.scatter(sub['tn_x_gas'], sub['fr_spot'], s=1, alpha=0.2, color='purple')
ax.set_xlabel('max(Thermal Need, 0) × Gas Price')
ax.set_ylabel('FR Spot')
ax.set_title(f'Feature interaction (r={r_tn_gas:.3f} vs gas={r_gas:.3f}, tn={r_tn:.3f})')

# UK version
mask_uk = train['uk_gas'].notna()
sub_uk = train[mask_uk].copy()

sub_uk['tn_bin'] = pd.cut(sub_uk['uk_thermal_need'], bins=15)
sub_uk['gas_bin'] = pd.cut(sub_uk['uk_gas'], bins=10)
pivot_uk = sub_uk.pivot_table(values='uk_spot', index='gas_bin', columns='tn_bin', aggfunc='mean')

ax = axes[1, 0]
sns.heatmap(pivot_uk, ax=ax, cmap='RdYlBu_r', annot=True, fmt='.0f', annot_kws={'size': 6})
ax.set_title('UK Spot moyen = f(Thermal Need, Gas Price)')
ax.set_xlabel('Thermal Need (MW)')
ax.set_ylabel('Gas Price (EUR/MWh)')

# UK wind × gas interaction
sub_uk['wind_gap'] = sub_uk['uk_load_f'] - sub_uk['uk_wind_f']  # what wind doesn't cover
sub_uk['wind_gap_x_gas'] = sub_uk['wind_gap'].clip(lower=0) * sub_uk['uk_gas']
r_wg = sub_uk['wind_gap_x_gas'].corr(sub_uk['uk_spot'])

ax = axes[1, 1]
ax.scatter(sub_uk['wind_gap_x_gas'], sub_uk['uk_spot'], s=1, alpha=0.2, color='#b2182b')
ax.set_xlabel('(Load - Wind) × Gas Price')
ax.set_ylabel('UK Spot')
ax.set_title(f'UK: Wind Gap × Gas Price (r={r_wg:.3f})')

# Comparison bar chart : individual vs interaction features
ax = axes[1, 2]
feats = {
    'fr_gas': sub['fr_gas'].corr(sub['fr_spot']),
    'fr_thermal_need': sub['fr_thermal_need'].corr(sub['fr_spot']),
    'fr_spark': sub['fr_spark'].corr(sub['fr_spot']),
    'tn_pos × gas': r_tn_gas,
    'uk_gas': sub_uk['uk_gas'].corr(sub_uk['uk_spot']),
    'uk_thermal_need': sub_uk['uk_thermal_need'].corr(sub_uk['uk_spot']),
    'uk_spark': sub_uk['uk_spark'].corr(sub_uk['uk_spot']),
    'uk_wind_gap × gas': r_wg,
}
colors = ['#2166ac']*4 + ['#b2182b']*4
ax.barh(range(len(feats)), list(feats.values()), color=colors)
ax.set_yticks(range(len(feats)))
ax.set_yticklabels(feats.keys(), fontsize=9)
ax.set_xlabel('Corrélation avec Spot')
ax.set_title('Features individuelles vs interactions')
for i, v in enumerate(feats.values()):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 3. Pourquoi juillet-août sont si difficiles à prédire

Le RMSE du naive 24h est 3-4x plus élevé en été qu'en hiver. Pourquoi ? Et comment mieux prédire ces mois ?

In [ ]:
# Séparer été (JJA) vs reste
summer = train[dt.dt.month.isin([7, 8])].copy()
winter = train[dt.dt.month.isin([12, 1, 2])].copy()
rest = train[~dt.dt.month.isin([7, 8])].copy()

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 3a. Distribution des prix été vs hiver
axes[0, 0].hist(summer['fr_spot'], bins=80, alpha=0.5, label=f'Été (std={summer["fr_spot"].std():.0f})', 
                density=True, color='#d73027')
axes[0, 0].hist(winter['fr_spot'], bins=80, alpha=0.5, label=f'Hiver (std={winter["fr_spot"].std():.0f})', 
                density=True, color='#2166ac')
axes[0, 0].set_title('FR: Distribution des prix Été vs Hiver')
axes[0, 0].set_xlabel('EUR/MWh')
axes[0, 0].legend()

# 3b. Volatilité intra-journalière (range max-min par jour)
train['date'] = dt.dt.date
daily_range = train.groupby('date').agg(
    fr_range=('fr_spot', lambda x: x.max() - x.min()),
    uk_range=('uk_spot', lambda x: x.max() - x.min()),
    month=('datetime_CET', lambda x: x.dt.month.iloc[0])
)

monthly_range = daily_range.groupby('month').agg(
    fr_mean=('fr_range', 'mean'),
    uk_mean=('uk_range', 'mean')
)
x = np.arange(12)
axes[0, 1].bar(x - 0.2, monthly_range['fr_mean'], 0.4, label='FR', color='#2166ac')
axes[0, 1].bar(x + 0.2, monthly_range['uk_mean'], 0.4, label='UK', color='#b2182b')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(range(1, 13))
axes[0, 1].set_title('Range intra-journalier moyen (max - min) par mois')
axes[0, 1].set_ylabel('EUR/MWh')
axes[0, 1].legend()

# 3c. En été : nuclear capacity est la clé
axes[0, 2].scatter(summer['fr_nuclear_avcap_f'], summer['fr_spot'], s=3, alpha=0.3, color='#d73027', label='Été')
axes[0, 2].scatter(winter['fr_nuclear_avcap_f'], winter['fr_spot'], s=3, alpha=0.3, color='#2166ac', label='Hiver')
axes[0, 2].set_xlabel('FR Nuclear Capacity (MW)')
axes[0, 2].set_ylabel('FR Spot')
axes[0, 2].set_title('Nucléaire vs Prix : relation différente été/hiver')
axes[0, 2].legend()

# 3d. Erreur du naive 24h — corrélation avec les features en été
summer['naive_error_fr'] = (summer['fr_spot'] - summer['fr_spot_la']).abs()
summer['naive_error_uk'] = (summer['uk_spot'] - summer['uk_spot_la']).abs()

# Quelles features expliquent les grosses erreurs du naive ?
error_corr_feats = ['fr_nuclear_avcap_f', 'fr_wind_f', 'fr_solar_f', 'fr_load_f',
                    'fr_hydro_res_f', 'fr_river_temp_rhone_lyon_f', 'fr_thermal_need']
corrs = {f: summer['naive_error_fr'].corr(summer[f]) for f in error_corr_feats if f in summer.columns}
corrs = dict(sorted(corrs.items(), key=lambda x: abs(x[1]), reverse=True))

axes[1, 0].barh(range(len(corrs)), list(corrs.values()), 
                color=['#d73027' if v < 0 else '#2166ac' for v in corrs.values()])
axes[1, 0].set_yticks(range(len(corrs)))
axes[1, 0].set_yticklabels(corrs.keys(), fontsize=9)
axes[1, 0].set_xlabel('Corrélation')
axes[1, 0].set_title('Quoi corrèle avec l\'erreur du naive en été (FR)')
axes[1, 0].axvline(0, color='black', linewidth=0.5)

# 3e. Solar duck curve ? Prix par heure en été
summer_hourly = summer.groupby(summer['datetime_CET'].dt.hour).agg(
    fr_price=('fr_spot', 'mean'),
    fr_solar=('fr_solar_f', 'mean'),
    fr_wind=('fr_wind_f', 'mean'),
)
ax = axes[1, 1]
ax.bar(summer_hourly.index, summer_hourly['fr_price'], color='steelblue', alpha=0.7, label='Prix moyen')
ax2 = ax.twinx()
ax2.plot(summer_hourly.index, summer_hourly['fr_solar'], color='orange', linewidth=2, label='Solar moyen')
ax.set_title('FR Été : Duck Curve — Solar écrase les prix l\'après-midi')
ax.set_xlabel('Heure')
ax.set_ylabel('Prix (EUR/MWh)')
ax2.set_ylabel('Solar (MW)', color='orange')
ax.legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)

# 3f. Transition d'un jour à l'autre en été (ce qui rend le naive mauvais)
summer['delta_nuclear_24h'] = summer['fr_nuclear_avcap_f'].diff(24)
mask_big_delta = summer['delta_nuclear_24h'].abs() > 2000
axes[1, 2].scatter(summer['delta_nuclear_24h'], summer['naive_error_fr'], s=3, alpha=0.3, color='#d73027')
axes[1, 2].set_xlabel('Δ Nuclear 24h (MW)')
axes[1, 2].set_ylabel('|Naive Error| FR (EUR/MWh)')
axes[1, 2].set_title('Changement nucléaire 24h → Erreur de prédiction')
r = summer['delta_nuclear_24h'].corr(summer['naive_error_fr'])
axes[1, 2].text(0.05, 0.95, f'r={r:.3f}', transform=axes[1, 2].transAxes, fontsize=11)

plt.tight_layout()
plt.show()

print(f'\n--- Été vs Hiver ---')
print(f'FR Spot std — Été: {summer["fr_spot"].std():.1f}, Hiver: {winter["fr_spot"].std():.1f}')
print(f'UK Spot std — Été: {summer["uk_spot"].std():.1f}, Hiver: {winter["uk_spot"].std():.1f}')
print(f'FR Nuclear moyen — Été: {summer["fr_nuclear_avcap_f"].mean():.0f} MW, Hiver: {winter["fr_nuclear_avcap_f"].mean():.0f} MW')

---
## 4. Nucléaire FR — Seuils critiques et dynamique

Le nucléaire est LE driver FR. Creusons : à quel seuil ça devient critique ? Quel est le lag optimal ?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 4a. Relation non-linéaire : binned mean avec bandes de confiance
nuke_bins = pd.cut(train['fr_nuclear_avcap_f'], bins=25)
nuke_binned = train.groupby(nuke_bins, observed=True)['fr_spot'].agg(['mean', 'median', 'std', 'count'])
bin_centers = [(b.left + b.right)/2 for b in nuke_binned.index]

ax = axes[0, 0]
ax.plot(bin_centers, nuke_binned['mean'], 'o-', color='#d73027', linewidth=2, label='Moyenne')
ax.plot(bin_centers, nuke_binned['median'], 's--', color='#2166ac', linewidth=2, label='Médiane')
ax.fill_between(bin_centers, 
                nuke_binned['mean'] - nuke_binned['std']/np.sqrt(nuke_binned['count']),
                nuke_binned['mean'] + nuke_binned['std']/np.sqrt(nuke_binned['count']),
                alpha=0.2, color='#d73027')
ax.set_xlabel('FR Nuclear Capacity (MW)')
ax.set_ylabel('FR Spot (EUR/MWh)')
ax.set_title('Relation non-linéaire : Nuclear → Prix')
ax.legend()

# 4b. Dérivée : à quel seuil le prix explose ?
ax = axes[0, 1]
slopes = np.diff(nuke_binned['mean'].values) / np.diff(bin_centers)
slope_centers = [(bin_centers[i] + bin_centers[i+1])/2 for i in range(len(slopes))]
ax.bar(slope_centers, slopes, width=(bin_centers[1]-bin_centers[0])*0.8, color='steelblue')
ax.set_xlabel('Nuclear Capacity (MW)')
ax.set_ylabel('d(Prix)/d(Nuclear) — EUR/MWh par MW')
ax.set_title('Sensibilité du prix au nucléaire (pente)')
ax.axhline(0, color='black', linewidth=0.5)

# 4c. Nuclear change vs prix change (dynamique)
train['nuke_change_24h'] = train['fr_nuclear_avcap_f'].diff(24)
train['price_change_24h'] = train['fr_spot'].diff(24)

ax = axes[0, 2]
ax.scatter(train['nuke_change_24h'], train['price_change_24h'], s=1, alpha=0.2)
ax.set_xlabel('Δ Nuclear 24h (MW)')
ax.set_ylabel('Δ FR Spot 24h (EUR/MWh)')
r = train['nuke_change_24h'].corr(train['price_change_24h'])
ax.set_title(f'Variation nucléaire → Variation prix (r={r:.3f})')
ax.axhline(0, color='red', linewidth=0.5)
ax.axvline(0, color='red', linewidth=0.5)

# 4d. Nuclear × Hydro interaction
train['fr_hydro_total'] = train['fr_hydro_res_f'] + train['fr_hydro_ror_f']
train['nuke_plus_hydro'] = train['fr_nuclear_avcap_f'] + train['fr_hydro_total']
train['baseload_gap'] = train['fr_load_f'] - train['nuke_plus_hydro']

ax = axes[1, 0]
sc = ax.scatter(train['fr_nuclear_avcap_f'], train['fr_hydro_total'], 
                c=train['fr_spot'], cmap='RdYlBu_r', s=2, alpha=0.4, vmin=-50, vmax=400)
ax.set_xlabel('Nuclear (MW)')
ax.set_ylabel('Hydro Total (MW)')
ax.set_title('Nuclear × Hydro → Prix (couleur)')
plt.colorbar(sc, ax=ax, label='FR Spot')

# 4e. Baseload gap (load - nuclear - hydro) vs price
r_bg = train['baseload_gap'].corr(train['fr_spot'])
r_tn = train['fr_thermal_need'].corr(train['fr_spot'])

ax = axes[1, 1]
ax.scatter(train['baseload_gap'], train['fr_spot'], s=1, alpha=0.2, color='purple')
ax.set_xlabel('Baseload Gap = Load - Nuclear - Hydro (MW)')
ax.set_ylabel('FR Spot')
ax.set_title(f'Baseload Gap vs Prix (r={r_bg:.3f} vs thermal_need r={r_tn:.3f})')

# 4f. Quel lag optimal pour nuclear → prix ?
ax = axes[1, 2]
lags = range(0, 49)
corrs = [train['fr_nuclear_avcap_f'].shift(l).corr(train['fr_spot']) for l in lags]
ax.plot(lags, corrs, 'o-', color='#2166ac', markersize=3)
best_lag = lags[np.argmin(corrs)]  # la plus négative
ax.axvline(best_lag, color='red', linestyle='--', label=f'Best lag = {best_lag}h (r={min(corrs):.3f})')
ax.set_xlabel('Lag (heures)')
ax.set_ylabel('Corrélation Nuclear → FR Spot')
ax.set_title('Lag optimal du nucléaire')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nBaseload gap (load - nuclear - hydro) : r={r_bg:.3f}')
print(f'Thermal need (residual - nuclear) :      r={r_tn:.3f}')
print(f'→ Baseload gap est {"meilleur" if abs(r_bg) > abs(r_tn) else "moins bon"} que thermal need')

---
## 5. UK Wind — Seuils de prix négatifs et non-linéarité

Le UK est gas+wind. Quand le vent souffle fort, les prix s'effondrent. À quel point ?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 5a. Wind penetration binned avec quantiles (non-linéarité)
wp_bins = pd.cut(train['uk_wind_pen']*100, bins=30)
wp_stats = train.groupby(wp_bins, observed=True)['uk_spot'].agg(
    ['mean', 'median', lambda x: x.quantile(0.1), lambda x: x.quantile(0.9)]
)
wp_stats.columns = ['mean', 'median', 'q10', 'q90']
bin_centers = [(b.left + b.right)/2 for b in wp_stats.index]

ax = axes[0, 0]
ax.fill_between(bin_centers, wp_stats['q10'], wp_stats['q90'], alpha=0.15, color='steelblue', label='Q10-Q90')
ax.plot(bin_centers, wp_stats['mean'], 'o-', color='#d73027', linewidth=2, markersize=3, label='Mean')
ax.plot(bin_centers, wp_stats['median'], 's--', color='#2166ac', linewidth=2, markersize=3, label='Median')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('UK Wind Penetration (%)')
ax.set_ylabel('UK Spot (EUR/MWh)')
ax.set_title('UK: Prix vs Wind Penetration (avec incertitude)')
ax.legend(fontsize=8)

# 5b. Probabilité de prix négatif vs wind penetration
wp_neg = train.groupby(wp_bins, observed=True)['uk_spot'].apply(lambda x: (x < 0).mean() * 100)
ax = axes[0, 1]
ax.bar(range(len(wp_neg)), wp_neg.values, color='#d73027', alpha=0.7)
ax.set_xticks(range(0, len(wp_neg), 3))
ax.set_xticklabels([f'{bin_centers[i]:.0f}%' for i in range(0, len(bin_centers), 3)], rotation=45, fontsize=8)
ax.set_xlabel('Wind Penetration (%)')
ax.set_ylabel('% de prix négatifs')
ax.set_title('UK: Probabilité de prix négatif vs éolien')

# 5c. Wind × Hour interaction (merit order shift par heure)
ax = axes[0, 2]
hours_to_show = [3, 9, 15, 19]  # nuit, matin, après-midi, soir
colors_h = ['#2166ac', '#4dac26', '#ff7f00', '#d73027']
for h, color in zip(hours_to_show, colors_h):
    mask_h = train['datetime_CET'].dt.hour == h
    wp_h = pd.cut(train.loc[mask_h, 'uk_wind_pen']*100, bins=15)
    mean_h = train.loc[mask_h].groupby(wp_h, observed=True)['uk_spot'].mean()
    centers_h = [(b.left + b.right)/2 for b in mean_h.index]
    ax.plot(centers_h, mean_h.values, 'o-', color=color, linewidth=1.5, markersize=3, label=f'{h}h')
ax.set_xlabel('UK Wind Penetration (%)')
ax.set_ylabel('UK Spot')
ax.set_title('Wind → Prix à différentes heures')
ax.legend(fontsize=8)

# 5d. FR wind effect (moins fort mais existe)
fr_wp_bins = pd.cut(train['fr_wind_pen']*100, bins=25)
fr_wp_stats = train.groupby(fr_wp_bins, observed=True)['fr_spot'].agg(['mean', 'median'])
fr_bc = [(b.left + b.right)/2 for b in fr_wp_stats.index]

ax = axes[1, 0]
ax.plot(fr_bc, fr_wp_stats['mean'], 'o-', color='#2166ac', linewidth=2, markersize=3, label='Mean')
ax.plot(fr_bc, fr_wp_stats['median'], 's--', color='gray', linewidth=1.5, markersize=3, label='Median')
ax.set_xlabel('FR Wind Penetration (%)')
ax.set_ylabel('FR Spot')
ax.set_title('FR: Wind Penetration vs Prix')
ax.legend()

# 5e. Wind ramp (variation rapide) → impact sur prix
train['uk_wind_ramp_3h'] = train['uk_wind_f'].diff(3)
ramp_bins = pd.cut(train['uk_wind_ramp_3h'], bins=20)
ramp_price = train.groupby(ramp_bins, observed=True)['uk_spot'].mean()
ramp_centers = [(b.left + b.right)/2 for b in ramp_price.index]

ax = axes[1, 1]
ax.plot(ramp_centers, ramp_price.values, 'o-', color='#4dac26', linewidth=2, markersize=4)
ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('UK Wind Ramp 3h (MW)')
ax.set_ylabel('UK Spot moyen')
ax.set_title('Rampe éolienne 3h → Impact prix UK')

# 5f. Combined UK: wind + gas capacity → prix
train['uk_dispatchable'] = train['uk_gas_avcap_f'] + train['uk_nuclear_avcap_f'] + train.get('uk_biomass_avcap_f', 0)
train['uk_supply_ratio'] = (train['uk_wind_f'] + train['uk_dispatchable']) / train['uk_load_f'].clip(lower=1)

sr_bins = pd.cut(train['uk_supply_ratio'], bins=25)
sr_price = train.groupby(sr_bins, observed=True)['uk_spot'].agg(['mean', 'median'])
sr_centers = [(b.left + b.right)/2 for b in sr_price.index]

ax = axes[1, 2]
ax.plot(sr_centers, sr_price['mean'], 'o-', color='#b2182b', linewidth=2, markersize=3)
ax.axvline(1.0, color='black', linewidth=1, linestyle='--', label='Supply = Demand')
ax.set_xlabel('UK Supply Ratio (Wind+Gas+Nuclear+Bio) / Load')
ax.set_ylabel('UK Spot moyen')
ax.set_title('UK: Ratio offre/demande → Prix')
ax.legend()

plt.tight_layout()
plt.show()

---
## 6. Régimes de prix — Crise 2022 vs Normalisation

Le train contient un changement de régime massif. Comment le gérer ?

In [ ]:
# Découper en 4 périodes
periods = {
    'Crise (Jul-Dec 2022)': (dt >= '2022-07-01') & (dt < '2023-01-01'),
    'Transition (Jan-Jun 2023)': (dt >= '2023-01-01') & (dt < '2023-07-01'),
    'Normal 1 (Jul-Dec 2023)': (dt >= '2023-07-01') & (dt < '2024-01-01'),
    'Normal 2 (Jan-Jun 2024)': (dt >= '2024-01-01') & (dt <= '2024-06-30'),
}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 6a. Distribution des prix par période
ax = axes[0, 0]
colors = ['#d73027', '#fc8d59', '#91bfdb', '#2166ac']
for (name, mask), color in zip(periods.items(), colors):
    ax.hist(train.loc[mask, 'fr_spot'], bins=60, alpha=0.5, label=name, density=True, color=color)
ax.set_title('FR Spot — Distribution par période')
ax.set_xlabel('EUR/MWh')
ax.legend(fontsize=8)

# 6b. Stats par période
ax = axes[0, 1]
stats = []
for name, mask in periods.items():
    sub = train[mask]
    stats.append({
        'Period': name.split('(')[0].strip(),
        'FR mean': sub['fr_spot'].mean(),
        'FR std': sub['fr_spot'].std(),
        'UK mean': sub['uk_spot'].mean(),
        'UK std': sub['uk_spot'].std(),
        'Nuclear': sub['fr_nuclear_avcap_f'].mean(),
    })
stats_df = pd.DataFrame(stats)

x = np.arange(4)
ax.bar(x - 0.2, stats_df['FR mean'], 0.4, yerr=stats_df['FR std'], capsize=3, label='FR', color='#2166ac', alpha=0.7)
ax.bar(x + 0.2, stats_df['UK mean'], 0.4, yerr=stats_df['UK std'], capsize=3, label='UK', color='#b2182b', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(stats_df['Period'], fontsize=9)
ax.set_title('Prix moyen (± std) par période')
ax.set_ylabel('EUR/MWh')
ax.legend()

# 6c. Corrélation spark-spot par période (la relation est-elle stable ?)
ax = axes[1, 0]
for (name, mask), color in zip(periods.items(), colors):
    sub = train[mask]
    m = sub['fr_spark'].notna()
    r = sub.loc[m, 'fr_spark'].corr(sub.loc[m, 'fr_spot'])
    ax.scatter(sub.loc[m, 'fr_spark'], sub.loc[m, 'fr_spot'], s=2, alpha=0.3, color=color, label=f'{name.split("(")[0].strip()} (r={r:.2f})')
ax.plot([0, 500], [0, 500], 'k--', linewidth=1)
ax.set_xlabel('FR Spark Spread')
ax.set_ylabel('FR Spot')
ax.set_title('Spark → Spot : même relation dans chaque régime ?')
ax.legend(fontsize=7)

# 6d. Poids optimal par période pour le RMSE du test
# Idée : si on pondère plus les données récentes, c'est mieux ?
ax = axes[1, 1]
ax.text(0.05, 0.95, 'Analyse: poids temporels pour le RMSE\n', transform=ax.transAxes, fontsize=12,
        verticalalignment='top', fontweight='bold')

text_lines = []
for name, mask in periods.items():
    sub = train[mask]
    fr_mean = sub['fr_spot'].mean()
    fr_std = sub['fr_spot'].std()
    n = mask.sum()
    text_lines.append(f'{name}:\n  FR: {fr_mean:.0f} ± {fr_std:.0f}  (n={n:,})\n')

# Test period estimates (from lagged prices in test)
test_fr_la = x_test['fr_spot_la'].mean()
test_uk_la = x_test['uk_spot_la'].mean()
text_lines.append(f'\nTest period (estimé via lagged):\n  FR: ~{test_fr_la:.0f}, UK: ~{test_uk_la:.0f}')
text_lines.append(f'\n→ Le test ressemble à Normal 2.')
text_lines.append(f'→ Pondérer les données récentes peut aider.')

ax.text(0.05, 0.85, '\n'.join(text_lines), transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace')
ax.axis('off')

plt.tight_layout()
plt.show()

print('\nStats par période :')
print(stats_df.to_string(index=False))

---
## 7. Congestion Interconnecteurs — Quand les marchés découplent

Quand les interconnecteurs FR-UK sont saturés, les prix divergent. C'est un signal fort.

In [ ]:
# ATC vs NTC → taux d'utilisation
train['fr_uk_atc'] = train[['atc_fr-uk-1_f', 'atc_fr-uk-2_f', 'atc_fr-uk-3_f']].sum(axis=1)
train['fr_uk_ntc'] = train[['ntc_fr-uk-1_f', 'ntc_fr-uk-2_f', 'ntc_fr-uk-3_f']].sum(axis=1)
train['uk_fr_atc'] = train[['atc_uk-fr-1_f', 'atc_uk-fr-2_f', 'atc_uk-fr-3_f']].sum(axis=1)
train['uk_fr_ntc'] = train[['ntc_uk-fr-1_f', 'ntc_uk-fr-2_f', 'ntc_uk-fr-3_f']].sum(axis=1)

# Utilization rate
train['fr_uk_util'] = 1 - (train['fr_uk_atc'] / train['fr_uk_ntc'].clip(lower=1))
train['uk_fr_util'] = 1 - (train['uk_fr_atc'] / train['uk_fr_ntc'].clip(lower=1))

# Lagged flows
fr_to_uk = train[['flow_fr-uk-1_la', 'flow_fr-uk-2_la', 'flow_fr-uk-3_la']].sum(axis=1)
uk_to_fr = train[['flow_uk-fr-1_la', 'flow_uk-fr-2_la', 'flow_uk-fr-3_la']].sum(axis=1)
train['fr_uk_net'] = fr_to_uk - uk_to_fr

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 7a. ATC total FR→UK time series
ax = axes[0, 0]
ax.plot(dt, train['fr_uk_atc'].rolling(168).mean(), linewidth=1, color='steelblue', label='FR→UK ATC')
ax.plot(dt, train['uk_fr_atc'].rolling(168).mean(), linewidth=1, color='coral', label='UK→FR ATC')
ax.set_title('Capacité disponible FR↔UK (rolling 7j)')
ax.set_ylabel('MW')
ax.legend(fontsize=8)

# 7b. Utilization rate vs spread
ax = axes[0, 1]
ax.scatter(train['fr_uk_util'], train['spread_fr_uk'], s=1, alpha=0.2, color='purple')
ax.set_xlabel('Utilization FR→UK (1 = plein)')
ax.set_ylabel('Spread FR - UK (EUR/MWh)')
ax.set_title('Congestion FR→UK vs divergence de prix')
ax.axhline(0, color='black', linewidth=0.5)

# 7c. |Spread| by congestion bucket
train['congestion_level'] = pd.cut(train['fr_uk_util'].clip(0, 1), 
                                    bins=[0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                                    labels=['<10%', '10-30%', '30-50%', '50-70%', '70-90%', '>90%'])
spread_by_cong = train.groupby('congestion_level', observed=True)['spread_fr_uk'].agg(
    ['mean', 'std', lambda x: (x.abs() > 50).mean() * 100]
)
spread_by_cong.columns = ['mean_spread', 'std_spread', 'pct_big_divergence']

ax = axes[0, 2]
ax.bar(range(len(spread_by_cong)), spread_by_cong['std_spread'], color='steelblue', alpha=0.7)
ax.set_xticks(range(len(spread_by_cong)))
ax.set_xticklabels(spread_by_cong.index, fontsize=9)
ax.set_xlabel('Congestion Level FR→UK')
ax.set_ylabel('Std du spread (EUR/MWh)')
ax.set_title('Plus de congestion = plus de volatilité du spread')

# 7d. Net flow vs price difference (lagged)
ax = axes[1, 0]
ax.scatter(train['fr_uk_net'], train['spread_fr_uk'], s=1, alpha=0.2)
ax.set_xlabel('Net flow FR→UK (MW, lagged)')
ax.set_ylabel('Spread FR - UK')
r = train['fr_uk_net'].corr(train['spread_fr_uk'])
ax.set_title(f'Flux net FR→UK vs Spread (r={r:.3f})')
ax.axhline(0, color='red', linewidth=0.5)
ax.axvline(0, color='red', linewidth=0.5)

# 7e. Cost of interconnector vs spread
train['fr_uk_cost_la'] = train[['cost_fr-uk-1_la', 'cost_fr-uk-2_la', 'cost_fr-uk-3_la']].mean(axis=1)
r_cost = train['fr_uk_cost_la'].corr(train['spread_fr_uk'])

ax = axes[1, 1]
mask_cost = train['fr_uk_cost_la'].notna() & (train['fr_uk_cost_la'] > 0)
ax.scatter(train.loc[mask_cost, 'fr_uk_cost_la'], train.loc[mask_cost, 'spread_fr_uk'], s=2, alpha=0.3)
ax.set_xlabel('Coût moyen interconnexion FR→UK (EUR/MWh, lagged)')
ax.set_ylabel('Spread FR - UK')
ax.set_title(f'Coût d\'accès au câble vs Spread (r={r_cost:.3f})')

# 7f. Résumé : quand les marchés sont couplés vs découplés
ax = axes[1, 2]
coupled = train[train['spread_fr_uk'].abs() < 10]  # prix presque identiques
decoupled = train[train['spread_fr_uk'].abs() > 50]  # forte divergence

compare_feats = ['fr_uk_atc', 'uk_wind_f', 'fr_nuclear_avcap_f', 'fr_load_f', 'uk_load_f']
coupled_means = coupled[compare_feats].mean()
decoupled_means = decoupled[compare_feats].mean()
ratio = (decoupled_means / coupled_means - 1) * 100

colors = ['#d73027' if v < 0 else '#2166ac' for v in ratio]
ax.barh(range(len(ratio)), ratio.values, color=colors)
ax.set_yticks(range(len(ratio)))
ax.set_yticklabels(ratio.index, fontsize=9)
ax.set_xlabel('% différence (découplé vs couplé)')
ax.set_title(f'Conditions : marchés couplés vs découplés')
ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f'\nMarchés couplés (|spread| < 10): {len(coupled)} heures ({100*len(coupled)/len(train):.1f}%)')
print(f'Marchés découplés (|spread| > 50): {len(decoupled)} heures ({100*len(decoupled)/len(train):.1f}%)')

---
## 8. Synthèse — Features candidates et priorités

In [ ]:
# Corrélations de toutes les features candidates avec les targets
candidates = {
    'fr_spark': train['fr_spark'],
    'uk_spark': train['uk_spark'],
    'fr_thermal_need': train['fr_thermal_need'],
    'uk_thermal_need': train['uk_thermal_need'],
    'baseload_gap': train['baseload_gap'],
    'fr_wind_pen': train['fr_wind_pen'],
    'uk_wind_pen': train['uk_wind_pen'],
    'fr_uk_net_flow': train['fr_uk_net'],
    'fr_uk_atc': train['fr_uk_atc'],
    'fr_uk_util': train['fr_uk_util'],
    'nuke_change_24h': train['nuke_change_24h'],
    'uk_wind_ramp_3h': train['uk_wind_ramp_3h'],
    'spread_fr_uk_la': train['fr_spot_la'] - train['uk_spot_la'],
    'fr_uk_cost_la': train['fr_uk_cost_la'],
}

print('='*70)
print('FEATURES CANDIDATES — Corrélations avec les targets')
print('='*70)
print(f'{"Feature":<25} {"r(FR spot)":>12} {"r(UK spot)":>12}')
print('-'*50)
for name, series in candidates.items():
    r_fr = series.corr(train['fr_spot'])
    r_uk = series.corr(train['uk_spot'])
    print(f'{name:<25} {r_fr:>12.3f} {r_uk:>12.3f}')

print(f"\n{'='*70}")
print('RECOMMANDATIONS POUR LE FEATURE ENGINEERING')
print('='*70)
print("""
HAUTE PRIORITÉ (impact fort attendu) :
  1. spark_spread (r=0.89/0.88) — mais forward-fill nécessaire
  2. thermal_need FR (r=0.675) et baseload_gap (inclut hydro)
  3. uk_wind_penetration (r=-0.36) — non-linéaire, seuils importants
  4. nuclear_change_24h — capture les transitions
  5. wind_ramp_3h UK — rampes rapides = impact fort

MOYENNE PRIORITÉ (contexte marché) :
  6. interconnector utilization/ATC/flows
  7. spread FR-UK lagged
  8. interconnector cost lagged

POINTS D'ATTENTION :
  - Spark spread a un biais horaire (nuit: spot << spark)
  - Régime crise 2022 vs normal → pondérer les données récentes
  - Été = danger max pour RMSE (range intra-jour 3x plus élevé)
  - Nuclear × Hydro interaction importante pour FR
  - UK: wind penetration est THE feature, non-linéaire
""")